In [6]:
import numpy as np
import torch


# -------------------------
# 1D GRF sampler (periodic)
# -------------------------
def sample_grf_1d(Nx, L=1.0, alpha=2.0, tau=3.0, sigma=0.218, k_cut=8, rng=None):
    """
    Periodic 1D Gaussian random field on Nx points over [0, L).
    Power spectrum ~ (tau^2 + k^2)^(-alpha/2).
    Returns u0 of shape (Nx,).
    """
    if rng is None:
        rng = np.random.default_rng()

    # Fourier wavenumbers (cycles per domain) -> angular uses 2π k / L
    k = np.fft.fftfreq(Nx, d=L / Nx)  # cycles per unit length
    lam = (tau**2 + (2.0 * np.pi * k) ** 2) ** (-alpha / 2.0)

    # Band-limit to match 16-res
    lam[np.abs(k) > k_cut] = 0.0
    
    # complex normal coefficients
    z = rng.normal(size=Nx) + 1j * rng.normal(size=Nx)
    u_hat = z * np.sqrt(lam)

    # real field
    u = np.fft.ifft(u_hat).real

    # normalize and scale
    u = (u - u.mean()) / (u.std() + 1e-12)
    u = sigma * u
    return u.astype(np.float64)



# ---------------------------------------------------
# Viscous Burgers solver with periodic FV (LLF flux)
# u_t + (1/2 u^2)_x = nu u_xx
# ---------------------------------------------------
def solve_viscous_burgers_fv_llf(u0, nu=1e-1, L=1.0, t_max=1.0,
                                Nx=100, Nt_out=100, CFL=0.4, safety=0.2):
    """
    Returns snapshots u(t_j, x_i) with shape (Nt_out, Nx), on [0,L) periodic grid.
    Uses internal stable dt, then records Nt_out evenly spaced snapshots.
    """
    assert u0.shape == (Nx,)

    dx = L / Nx
    u = u0.copy()

    # estimate stable dt
    max_u = max(np.max(np.abs(u)), 1e-8)
    dt_conv = CFL * dx / max_u
    dt_diff = 0.5 * dx * dx / max(nu, 1e-12)
    dt_base = min(dt_conv, dt_diff, t_max / max(Nt_out - 1, 1))
    dt = safety * dt_base

    n_steps = int(np.ceil(t_max / dt))
    dt = t_max / n_steps  # hit t_max exactly

    # indices where we save snapshots
    save_times = np.linspace(0.0, t_max, Nt_out)
    save_idx = 0
    t = 0.0

    snapshots = np.empty((Nt_out, Nx), dtype=np.float64)
    snapshots[0] = u.copy()
    save_idx = 1

    for step in range(n_steps):
        u_prev = u

        # periodic neighbors
        u_right = np.roll(u_prev, -1)
        u_left  = np.roll(u_prev,  1)

        # flux f(u) = 0.5 u^2
        f = 0.5 * u_prev**2
        f_right = np.roll(f, -1)

        # local wave speed a_{i+1/2} = max(|u_i|, |u_{i+1}|)
        a = np.maximum(np.abs(u_prev), np.abs(u_right))

        # LLF flux: F_{i+1/2} = 0.5(f_i + f_{i+1}) - 0.5 a (u_{i+1}-u_i)
        F_iphalf = 0.5 * (f + f_right) - 0.5 * a * (u_right - u_prev)
        F_imhalf = np.roll(F_iphalf, 1)

        # convection + diffusion
        convection = -(F_iphalf - F_imhalf) / dx
        u_xx = (u_right - 2.0 * u_prev + u_left) / (dx * dx)
        diffusion = nu * u_xx

        # step
        u = u_prev + dt * (convection + diffusion)
        t += dt

        if not np.all(np.isfinite(u)):
            raise RuntimeError(f"Blow-up (non-finite) at step {step}, t={t:.4e}. Try larger nu or smaller safety.")

        # save snapshots when passing save_times
        while save_idx < Nt_out and t + 1e-15 >= save_times[save_idx]:
            snapshots[save_idx] = u.copy()
            save_idx += 1

        if save_idx >= Nt_out:
            break

    # if any remaining (numerical corner), fill with last
    while save_idx < Nt_out:
        snapshots[save_idx] = u.copy()
        save_idx += 1

    return snapshots


def build_and_save(split_name, N, Nx=100, Nt_out=100, nu=1e-1, seed=0, out_path="burgers_train_100.pt"):
    rng = np.random.default_rng(seed)

    X = np.empty((N, Nx), dtype=np.float64)          # ICs
    Y = np.empty((N, Nt_out, Nx), dtype=np.float64)  # trajectories
    visc = np.full((N,), nu, dtype=np.float64)

    for i in range(N):
        #u0 = sample_grf_1d(Nx, alpha=2.0, tau=3.0, sigma=1.0, rng=rng)
        u0 = sample_grf_1d(Nx, alpha=2.0, tau=3.0, sigma=0.218, rng=rng)
        traj = solve_viscous_burgers_fv_llf(u0, nu=nu, Nx=Nx, Nt_out=Nt_out)

        X[i] = u0
        Y[i] = traj

        if (i + 1) % 50 == 0:
            print(f"[{split_name}] {i+1}/{N} done")

    data = {
        "x": torch.from_numpy(X),     # [N, Nx]
        "y": torch.from_numpy(Y),     # [N, Nt_out, Nx]
        "visc": torch.from_numpy(visc)  # [N]
    }
    torch.save(data, out_path)
    print(f"Saved {split_name} → {out_path}")
    print("Shapes:", data["x"].shape, data["y"].shape, data["visc"].shape)


if __name__ == "__main__":
    # Train
    build_and_save("train", N=800, Nx=100, Nt_out=100, nu=1e-1, seed=123,
                   out_path="burgers_train_100.pt")
    # Test
    build_and_save("test", N=400, Nx=100, Nt_out=100, nu=1e-1, seed=999,
                   out_path="burgers_test_100.pt")


[train] 50/800 done
[train] 100/800 done
[train] 150/800 done
[train] 200/800 done
[train] 250/800 done
[train] 300/800 done
[train] 350/800 done
[train] 400/800 done
[train] 450/800 done
[train] 500/800 done
[train] 550/800 done
[train] 600/800 done
[train] 650/800 done
[train] 700/800 done
[train] 750/800 done
[train] 800/800 done
Saved train → burgers_train_100.pt
Shapes: torch.Size([800, 100]) torch.Size([800, 100, 100]) torch.Size([800])
[test] 50/400 done
[test] 100/400 done
[test] 150/400 done
[test] 200/400 done
[test] 250/400 done
[test] 300/400 done
[test] 350/400 done
[test] 400/400 done
Saved test → burgers_test_100.pt
Shapes: torch.Size([400, 100]) torch.Size([400, 100, 100]) torch.Size([400])


In [9]:
%cd /home/jw3275/neuraloperator/neuralop/data/datasets/data
import torch
d = torch.load("burgers_test_100.pt", map_location="cpu")
print(type(d), d.keys())
print(d["x"].shape, d["y"].shape, d["visc"].shape)
print(d["x"][0][:5], d["y"][0,0,:5])


/home/jw3275/neuraloperator/neuralop/data/datasets/data
<class 'dict'> dict_keys(['x', 'y', 'visc'])
torch.Size([400, 100]) torch.Size([400, 100, 100]) torch.Size([400])
tensor([-0.0179, -0.0599, -0.0933, -0.1179, -0.1352], dtype=torch.float64) tensor([-0.0179, -0.0599, -0.0933, -0.1179, -0.1352], dtype=torch.float64)


In [8]:
%cd /home/jw3275/neuraloperator/neuralop/data/datasets/data
import torch
d = torch.load("burgers_test_100_v_01.pt", map_location="cpu")
print(type(d), d.keys())
print(d["x"].shape, d["y"].shape, d["visc"].shape)
print(d["x"][0][:5], d["y"][0,0,:5])


/home/jw3275/neuraloperator/neuralop/data/datasets/data
<class 'dict'> dict_keys(['x', 'y', 'visc'])
torch.Size([400, 100]) torch.Size([400, 100, 100]) torch.Size([400])
tensor([-0.0179, -0.0599, -0.0933, -0.1179, -0.1352], dtype=torch.float64) tensor([-0.0179, -0.0599, -0.0933, -0.1179, -0.1352], dtype=torch.float64)
